# Практическое задание 1 по теме «Разведочный анализ данных (EDA)»

## Учебный датасет

Ниже мы создаём небольшой искусственный датасет о клиентах телеком-компании.  
Целевая переменная: **churn** — ушёл клиент или нет.

В таблице специально добавлены:
- пропуски;
- ошибочные значения;
- категориальные признаки;
- выбросы.

Это позволит потренировать основные шаги EDA и увидеть, как каждый шаг подготовки данных помогает перед обучением модели.

Во всех следующих заданиях полезно держать в голове один вопрос:  
**поможет ли эта проверка сделать данные более понятными и более пригодными для модели?**


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

np.random.seed(42)

n = 180

age = np.random.randint(18, 70, size=n).astype(float)
income = np.random.normal(65000, 18000, size=n)
months = np.random.randint(1, 72, size=n)
calls = np.random.poisson(95, size=n).astype(float)
internet = np.random.choice(["DSL", "Fiber", "None"], size=n, p=[0.35, 0.50, 0.15])
support = np.random.poisson(2.2, size=n).astype(float)
contract = np.random.choice(["Monthly", "Yearly"], size=n, p=[0.72, 0.28])

# Вероятность ухода зависит от нескольких факторов
score = (
    0.035 * support
    + 0.000018 * (85000 - income)
    + 0.012 * (12 - np.minimum(months, 12))
    + 0.25 * (contract == "Monthly").astype(float)
    + 0.12 * (internet == "Fiber").astype(float)
)
prob = 1 / (1 + np.exp(-(score - 0.5)))
churn = (np.random.rand(n) < prob).astype(int)

df = pd.DataFrame({
    "age": age,
    "income": income.round(0),
    "months_with_company": months,
    "monthly_calls": calls,
    "internet_type": internet,
    "support_calls": support,
    "contract_type": contract,
    "churn": churn
})

# Добавим пропуски
for idx in np.random.choice(df.index, 10, replace=False):
    df.loc[idx, "income"] = np.nan

for idx in np.random.choice(df.index, 8, replace=False):
    df.loc[idx, "support_calls"] = np.nan

for idx in np.random.choice(df.index, 5, replace=False):
    df.loc[idx, "internet_type"] = np.nan

# Добавим ошибки и выбросы
df.loc[5, "age"] = -3
df.loc[17, "income"] = 250000
df.loc[28, "monthly_calls"] = 450
df.loc[44, "income"] = -10000
df.loc[63, "internet_type"] = "fiber"      # ошибка в регистре
df.loc[79, "contract_type"] = "month-to-month"
df.loc[101, "monthly_calls"] = np.nan

csv_path = "eda_practice_dataset_1.csv"
df.to_csv(csv_path, index=False)

print(f"Файл создан: {csv_path}")
df.head()


---

# Задание 1. Загрузка данных и первое знакомство

## Что нужно сделать
1. Загрузите датасет из файла `eda_practice_dataset_1.csv`.
2. Выведите первые 5 строк.
3. Посмотрите, сколько в таблице строк и столбцов.

## Зачем это нужно
Это самый первый шаг EDA. Он помогает понять:
- как выглядит таблица;
- какие признаки вообще есть;
- с чем предстоит работать дальше.

## Подсказки по Pandas
- `pd.read_csv(...)` — загрузка таблицы из CSV;
- `df.head()` — первые строки;
- `df.shape` — размер таблицы в формате `(число_строк, число_столбцов)`.

## Решение

In [ ]:
# Поместите Ваш код сюда


# Задание 2. Типы данных и названия признаков

## Что нужно сделать
1. Выведите список столбцов.
2. Посмотрите типы данных всех столбцов.
3. Попробуйте ответить:
   - какие признаки числовые;
   - какие категориальные;
   - какой столбец является целевой переменной.

Запишите найденные ответы в текстовой секции ниже.

## Зачем это нужно
От типов признаков зависит дальнейшая обработка:
- числовые признаки можно описывать статистиками, строить гистограммы, вычислять корреляции;
- категориальные признаки нужно анализировать по частотам и строить для них one-hot-представление;
- целевая переменная нужна для построения модели.

## Подсказки по Pandas
- `df.columns` — список столбцов;
- `df.dtypes` — типы столбцов;
- `df.info()` — краткая сводка по таблице.

## Решение

In [ ]:
# Поместите Ваш код сюда


**Впишите свои ответы на впоросы сюда:**

...

# Задание 3. Поиск пропусков

## Что нужно сделать
1. Найдите, в каких столбцах есть пропущенные значения.
2. Подсчитайте количество пропусков в каждом столбце.
3. Определите, какие признаки требуют очистки.

## Зачем это нужно
Многие модели машинного обучения не умеют работать с `NaN`.
Если пропуски не обработать, обучение модели может завершиться ошибкой или дать плохой результат.

## Подсказки по Pandas
- `df.isna()` — таблица из True/False, показывающая, где есть пропуски;
- `df.isna().sum()` — количество пропусков по каждому столбцу;
- `df[df["income"].isna()]` — строки, где в столбце `income` пропуск.

## Решение

In [ ]:
# Поместите Ваш код сюда


# Задание 4. Проверка на некорректные значения

## Что нужно сделать
Проверьте, есть ли в таблице значения, которые выглядят неправдоподобно:
- отрицательный возраст;
- отрицательный доход;
- слишком большое число звонков в месяц;
- разные варианты записи одной и той же категории.

## Зачем это нужно
Такие ошибки искажают статистику и могут ухудшать качество модели.
Например, отрицательный возраст не имеет смысла и может повлиять на среднее значение и распределение признака.

## Подсказки по Pandas
- `df[df["age"] < 0]` — отбор строк по условию;
- `df["internet_type"].unique()` — все уникальные значения столбца;
- `df["contract_type"].unique()` — полезно для поиска опечаток в категориях.

## Решение


In [ ]:
# Поместите Ваш код сюда


# Задание 5. Простая очистка данных

## Что нужно сделать
Создайте новую таблицу `df_clean`, в которой:
1. исправлены ошибки в категориальных столбцах:
   - `fiber` → `Fiber`
   - `month-to-month` → `Monthly`
2. пропуски в категориальном столбце `internet_type` заполнены значением `"Unknown"`;
3. некорректные отрицательные значения в `age` и `income` заменены на `NaN`;
4. пропуски в числовых столбцах `income`, `support_calls`, `monthly_calls`, `age` заполнены медианой соответствующего столбца.

## Зачем это нужно
Это первое настоящее решение по результатам EDA.

Что мы увидели раньше:
- в категориальных столбцах есть разные записи одной и той же категории;
- в `internet_type` есть пропуски;
- в числовых признаках есть невозможные значения и `NaN`.

Что делаем теперь:
- приводим категории к единому виду;
- явно помечаем отсутствие информации как `"Unknown"`;
- заменяем ошибочные значения на пропуски;
- затем заполняем числовые пропуски медианой.

## Почему это полезно для модели
- одинаковые категории не будут ошибочно считаться разными;
- `NaN` не помешают обучению;
- медиана менее подвержена искажениями от выбросов, чем среднее;
- после очистки таблицу уже можно безопасно анализировать и передавать в модель.

## Подсказки по Pandas
- `df["col"].replace({...})` — замена значений;
- `df["col"].fillna(...)` — заполнение пропусков;
- `df.loc[условие, "col"] = ...` — замена значений по условию;
- `df["col"].median()` — медиана столбца.

## Решение


In [ ]:
# Поместите Ваш код сюда


# Задание 6. Описание числовых признаков

## Что нужно сделать
Для очищенной таблицы:
1. выведите описательную статистику числовых признаков;
2. сравните среднее и медиану хотя бы для двух столбцов;
3. попробуйте сделать вывод, где распределение может быть несимметричным или содержать выбросы.

## Зачем это нужно
После базовой очистки полезно проверить, что именно осталось в данных.

Если среднее и медиана заметно различаются, это часто означает одно из двух:
- распределение несимметрично;
- есть выбросы, которые тянут среднее вверх или вниз.

## Почему это полезно для модели
Такая проверка помогает решить:
- нужно ли дополнительно работать с выбросами;
- стоит ли применять масштабирование или преобразование признака;
- не будет ли отдельный столбец слишком сильно влиять на модель.

## Подсказки по Pandas
- `df.describe()` — статистика по числовым столбцам;
- `df["col"].mean()` — среднее;
- `df["col"].median()` — медиана.

## Решение


In [ ]:
# Поместите Ваш код сюда


# Задание 7. Визуальный анализ распределений и работа с выбросом

## Что нужно сделать
1. Постройте гистограмму признака `income`.
2. Постройте boxplot для признака `income`.
3. Постройте boxplot для признака `monthly_calls`.
4. По графикам объясните, есть ли выброс в `income`.
5. Обсудите варианты, что можно сделать с выбросом:
   - удалить строку;
   - ограничить слишком большое значение сверху;
   - оставить как есть.
6. Выберите один вариант и выполните преобразование.

## Зачем это нужно
На прошлом шаге мы увидели общие статистики.  
Теперь нужно визуально проверить, **не скрывается ли за ними экстремальное значение**.

Если признак содержит очень большой выброс, он может:
- искажать среднее;
- влиять на масштаб признака;
- мешать некоторым моделям, особенно линейным.

## Рекомендуемое решение
Для этого учебного примера удобно **не удалять строку**, а **ограничить слишком большие значения сверху**.  
Такой подход:
- сохраняет все наблюдения;
- уменьшает влияние экстремального значения;
- остаётся простым для понимания.

В качестве простого правила можно взять верхнюю границу по IQR:
`Q3 + 1.5 * IQR`, где Q3 - третий квартиль данных, IQR = Q3-Q1 - межквартильный разброс.

## Подсказки
- Для столбца pandas можно вызвать `.hist()`;
- boxplot можно построить через `plt.boxplot(...)`;
- квартиль 25%: `df["col"].quantile(0.25)`;
- квартиль 75%: `df["col"].quantile(0.75)`;
- ограничение значений: `df["col"] = df["col"].clip(upper=...)`.

## Решение


In [ ]:
# Поместите Ваш код сюда


# Задание 8. Анализ категориальных признаков и целевой переменной

## Что нужно сделать
1. Посчитайте, сколько раз встречается каждое значение в `internet_type`.
2. Посчитайте, сколько раз встречается каждое значение в `contract_type`.
3. Посчитайте распределение целевой переменной `churn`.

## Зачем это нужно
После очистки категориальных столбцов полезно проверить, какой стала структура выборки:
- не осталось ли редких или неожиданных категорий;
- появилась ли отдельная категория `"Unknown"`;
- сбалансирована ли целевая переменная.

## Почему это полезно для модели
- редкие категории иногда создают проблемы при кодировании;
- дисбаланс классов влияет на выбор метрики;
- понимание распределения `churn` помогает правильно интерпретировать качество модели.

## Подсказки по Pandas
- `df["col"].value_counts()` — подсчёт количества значений;
- `df["col"].value_counts(normalize=True)` — доли.

## Решение


In [ ]:
# Поместите Ваш код сюда


# Задание 9. Связи между числовыми признаками

## Что нужно сделать
1. Постройте матрицу корреляций для числовых признаков.
2. Найдите признаки, которые сильнее всего связаны с `churn`.
3. Постройте диаграмму рассеяния `support_calls` против `income`.

## Зачем это нужно
После очистки данных можно переходить к поиску полезных закономерностей.

Корреляции и scatter plot помогают увидеть:
- какие признаки связаны с целевой переменной;
- какие признаки похожи друг на друга;
- есть ли пары признаков, для которых наблюдается заметная зависимость.

## Почему это полезно для модели
- признаки, связанные с `churn`, могут оказаться полезными для обучения;
- если два признака почти дублируют друг друга, это важно заметить заранее;
- scatter plot позволяет проверить, не осталось ли странных точек после очистки.

## Подсказки по Pandas
- `df.corr(numeric_only=True)` — матрица корреляций;
- `corr["churn"].sort_values(...)` — сортировка связей с целевой переменной;
- диаграмма рассеяния строится через `plt.scatter(...)`.

## Решение


In [ ]:
# Поместите Ваш код сюда


# Задание 10. Сравнение модели на «сыром» и очищенном датасете

## Что нужно сделать
1. Попробуйте обучить простую модель на исходном датасете `df`.
2. Затем обучите такую же модель на очищенном датасете `df_clean`.
3. Сравните результаты и сделайте вывод, зачем вообще нужен EDA.

## Зачем это нужно
Это итоговое задание, где EDA связывается с машинным обучением напрямую.

Логика такая:
- сначала пробуем обучить модель на данных «как есть»;
- затем используем данные после исправления пропусков, категорий и выброса;
- после этого сравниваем, изменилось ли обучение и качество модели.

## Почему это важно
Такой эксперимент показывает, что EDA — не украшение и не набор красивых графиков.  
Он помогает:
- сделать обучение модели возможным;
- уменьшить количество технических ошибок;
- получить более осмысленные признаки и более надёжный результат.

## Подсказки
- Для модели нужно отделить признаки `X` от целевой переменной `y`;
- категориальные признаки нужно перевести в one-hot представление, например при помощи функции `pd.get_dummies(...)`;
- сырые данные могут вызвать ошибку из-за `NaN`;
- очищенные данные обычно подходят для обучения лучше.

Если вы ещё не работали с моделями, используйте следующий шаблон:

1. Разделение данных:
```python
from sklearn.model_selection import train_test_split

X = df[['age', 'income']]
y = (df['income'] > 50000).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
```

2. Создание модели:
```python
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, y_train)
```

3. Оценка качества:
```python
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)
accuracy_score(y_test, y_pred)
```

## Решение


In [1]:
# Поместите Ваш код сюда


# Контрольные вопросы

1. Почему модель на сырых данных может не обучиться вообще?
2. Какую проблему решило заполнение `internet_type = "Unknown"`?
3. Почему отрицательные значения в `age` и `income` сначала лучше превратить в `NaN`, а уже потом заполнять?
4. Почему для заполнения числовых пропусков здесь выбрана медиана, а не среднее?
5. Почему выброс в `income` лучше было не игнорировать?
6. В каких ситуациях выброс лучше удалить, а в каких — ограничить?
7. Какие признаки из этого датасета кажутся наиболее полезными для предсказания `churn`?
8. Какие дополнительные шаги EDA можно было бы сделать, если бы курс был более продвинутым?

# Обзор того, что было сделано

**1. Посмотреть на таблицу** → понять структуру данных.  
**2. Найти пропуски и ошибки** → понять, что мешает анализу и модели.  
**3. Очистить данные** → сделать таблицу пригодной для дальнейшей работы.  
**4. Изучить распределения и выбросы** → понять, нужны ли дополнительные преобразования.  
**5. Проверить связи признаков и целевой переменной** → наметить полезные признаки для модели.  
**6. Сравнить модель до и после EDA** → убедиться, что разведочный анализ реально влияет на результат.